# EXP-005 — Thực nghiệm Random Forest

- **Ngày chạy lại** 18/11/2026
- **Trạng thái** Lần 1 dùng kết quả G3 đã lưu. Lần 2 đã chạy và xác minh đủ 30 fold.

## Lần 1 — Kết quả lịch sử G3-16/11/2026

Lần 1 dùng lại điểm Random Forest mặc định đã có từ G3. Mỗi bài toán có 15 lượt kiểm tra, mỗi lượt giữ riêng một người làm tập test. Kết quả này làm mốc so sánh với Lần 2.

Điểm trung bình trên 15 người cho mỗi bài toán.

| Bài toán | Macro-F1 | Balanced Accuracy |
|---|---:|---:|
| Nhị phân | 0,8735 | 0,8865 |
| Ba lớp | 0,6480 | 0,6700 |

Xem [điểm RF theo từng người](../../experiments/01_baselines/EXP-005_RF/results.csv) và [báo cáo tổng hợp G3](../../docs/G3_TONG_HOP.md).

## Lần 2 — Kiểm tra chạy lại và đối chiếu kết quả

Random Forest mặc định được chạy lại với 100 cây, seed 42, cùng bộ physiological68 và cách chia LOSO. Lượt này có hồ sơ xác minh riêng trong `outputs/exp005_rf_full/`.

## Đối chiếu

Ghép điểm theo bài toán và người test. Ghép dự đoán theo bài toán, người test và mã cửa sổ. Chênh lệch được tính bằng Lần 2 trừ Lần 1. Ngưỡng so sánh điểm là `1e-10`.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

here = Path.cwd()
repo = next(path for path in (here, *here.parents) if (path / "scripts/run_g3_baselines.py").is_file())
experiment = repo / "experiments/01_baselines/EXP-005_RF"
output = repo / "outputs/exp005_rf_full"
verification = json.loads((output / "verification.json").read_text(encoding="utf-8"))
assert verification["status"] == "passed" and verification["models"] == ["rf"]
assert verification["n_folds"] == 30 and verification["n_predictions"] == 12378

keys = ["task", "subject_id"]
metrics = ["macro_f1", "balanced_accuracy"]
g3 = pd.read_csv(experiment / "results.csv")
rerun = pd.read_csv(output / "verified_metrics_by_subject.csv")
points = g3[keys + metrics].merge(
    rerun[keys + metrics], on=keys, suffixes=("_g3", "_rerun"), validate="one_to_one"
)
assert len(g3) == len(rerun) == len(points) == 30

for metric in metrics:
    points[f"{metric}_delta"] = points[f"{metric}_rerun"] - points[f"{metric}_g3"]

columns = [f"{metric}_{part}" for metric in metrics for part in ("g3", "rerun", "delta")]
display(points.groupby("task")[columns].mean().round(12))
delta = points[[f"{metric}_delta" for metric in metrics]]
changed = ~np.isclose(delta, 0, atol=1e-10, rtol=0).all(axis=1)
print(f"Fold có điểm khác: {changed.sum()}/{len(points)}")
print(f"Chênh lệch tuyệt đối lớn nhất: {delta.abs().max().max():.3g}")
if changed.any():
    display(points.loc[changed])


In [ ]:
prediction_keys = ["task", "subject_id", "window_id"]
g3_predictions_path = repo / "outputs/g3_group_predictions_six_models_20261108/predictions.csv"
if g3_predictions_path.is_file():
    g3_predictions = pd.read_csv(g3_predictions_path).query("model == 'rf'")
    rerun_predictions = pd.read_csv(output / "predictions.csv")
    columns = prediction_keys + ["y_true", "y_pred"]
    predictions = g3_predictions[columns].merge(
        rerun_predictions[columns], on=prediction_keys,
        suffixes=("_g3", "_rerun"), validate="one_to_one"
    )
    assert len(predictions) == len(g3_predictions) == len(rerun_predictions) == 12378
    assert predictions["y_true_g3"].eq(predictions["y_true_rerun"]).all()
    changed_predictions = predictions["y_pred_g3"].ne(predictions["y_pred_rerun"])
    print(f"Dự đoán thay đổi: {changed_predictions.sum()}/{len(predictions)}")
else:
    print("Thiếu bảng dự đoán chi tiết G3 để đối chiếu từng cửa sổ.")


## Nhận xét

Ở lượt chạy lại đã xác minh, có **0/30 fold** thay đổi Macro-F1 hoặc Balanced Accuracy theo ngưỡng đối chiếu và **0/12.378 dự đoán** đổi nhãn. Hai lượt được lưu ở các nguồn khác nhau. Lần 1 là hồ sơ G3, còn Lần 2 là lần huấn luyện lại của EXP-005.

**Hai metric khớp kết quả G3 ở đủ 30 fold.**


## Bổ sung ngày 19/11/2026 — So sánh cấu hình và điểm train–test

Trạng thái: đã chạy và xác minh đủ 90 lượt LOSO trên 15 người cho hai bài toán.

Cấu hình A sử dụng Random Forest với 20 cây. A+B bổ sung giới hạn độ sâu tối đa là 8. A+B+C tiếp tục đặt `max_features=0.25`, tương ứng xét 25% số đặc trưng tại mỗi lần chia nhánh. Cả ba cấu hình dùng seed 42. Số cây được giảm để phù hợp tài nguyên máy, nên kết quả chỉ phản ánh các cấu hình đang khảo sát.

Mỗi lượt LOSO dùng dữ liệu của 14 người để huấn luyện và người còn lại để kiểm tra. Giá trị thiếu được điền bằng trung vị tính từ tập train. Điểm train và test được tính từ cùng mô hình đã huấn luyện, sau đó lấy trung bình qua 15 lượt.

| Bài toán | Cấu hình | Train Macro-F1 | Test Macro-F1 | Train BA | Test BA | Chênh lệch F1 train − test |
|---|---|---:|---:|---:|---:|---:|
| Nhị phân | A | 1,0000 | 0,8678 | 1,0000 | 0,8756 | 0,1322 |
| Nhị phân | A+B | 0,9999 | 0,8624 | 0,9999 | 0,8711 | 0,1375 |
| Nhị phân | A+B+C | 0,9997 | 0,8813 | 0,9997 | 0,8933 | 0,1184 |
| Ba lớp | A | 1,0000 | 0,6278 | 1,0000 | 0,6445 | 0,3722 |
| Ba lớp | A+B | 0,9977 | 0,6414 | 0,9966 | 0,6641 | 0,3563 |
| Ba lớp | A+B+C | 0,9969 | 0,6147 | 0,9955 | 0,6477 | 0,3822 |

### Nhận xét

Ở bài toán nhị phân, A+B+C đạt điểm test cao nhất trong ba cấu hình, với Macro-F1 0,8813 và Balanced Accuracy 0,8933. So với A, hai chỉ số tăng lần lượt 0,0134 và 0,0177; chênh lệch Macro-F1 giữa train và test giảm từ 0,1322 xuống 0,1184. Riêng việc thêm giới hạn độ sâu ở A+B làm điểm test giảm nhẹ.

Ở bài toán ba lớp, A+B đạt điểm test cao nhất với Macro-F1 0,6414 và Balanced Accuracy 0,6641. So với A, hai chỉ số tăng lần lượt 0,0136 và 0,0196. Khi thêm C, Macro-F1 giảm xuống 0,6147 và Balanced Accuracy còn 0,6477. Như vậy, thêm thay đổi cấu hình chưa chắc giúp mô hình tốt hơn ở cả hai bài toán.

Điểm train của các cấu hình đều gần 1, trong khi điểm test thấp hơn, đặc biệt ở bài toán ba lớp. Kết quả gợi ý cần xem xét overfitting và sự khác biệt giữa người tham gia; riêng khoảng cách train–test chưa xác định được nguyên nhân. Các mức cải thiện trên là chênh lệch quan sát được, chưa có kiểm định để kết luận ý nghĩa thống kê.

A+B+C phù hợp hơn cho nhị phân và A+B phù hợp hơn cho ba lớp trong phạm vi ba cấu hình đã chạy. Đây là khảo sát cấu hình cố định; quy trình chọn tham số bằng inner CV thuộc EXP-009.

Xem [notebook thực nghiệm đầy đủ](../../experiments/01_baselines/EXP-005_RF/experiment.ipynb), [cấu hình](../../experiments/01_baselines/EXP-005_RF/config_train_test.yaml) và [kết quả theo từng lượt](../../experiments/01_baselines/EXP-005_RF/results_train_test.csv).

![So sánh điểm train và test](../../experiments/01_baselines/EXP-005_RF/figures/rf_cumulative_train_test.png)



## Lần 4 — So sánh 20 cây và 100 cây -22/11/2026

Đã có kết quả đầy đủ cho cả hai lượt: mỗi lượt gồm 3 cấu hình × 2 bài toán × 15 người = **90 fold mô hình**. Phần này đọc kết quả đã lưu, không huấn luyện lại.

| Cấu hình | Độ sâu tối đa | Đặc trưng xét tại mỗi lần chia nhánh |
| --- | --- | --- |
| A | Không giới hạn | `sqrt` |
| A+B | 8 | `sqrt` |
| A+B+C | 8 | 25% số đặc trưng |

Mỗi cấu hình được đối chiếu giữa 20 và 100 cây. Giữ nguyên dữ liệu 68 đặc trưng, 15 người, seed 42, cách điền giá trị thiếu từ tập train và giao thức LOSO (14 người train, 1 người test). Hai hồ sơ chạy có cùng phiên bản thư viện và mã xử lý. Các giá trị mặc định `max_depth=None`, `max_features="sqrt"` ở lượt 20 cây được tính rõ khi kiểm tra cấu hình.

Điểm trong bảng là trung bình đều trên 15 người test, không phải gộp toàn bộ cửa sổ. **Δ = điểm 100 cây − điểm 20 cây**; số dương là tăng. BA là Balanced Accuracy. Train − test là khoảng cách giữa điểm trên dữ liệu đã học và điểm trên người giữ riêng.

In [1]:
from pathlib import Path
import json
import pandas as pd

here = Path.cwd()
comparison_repo = next(p for p in (here, *here.parents) if (p / 'scripts/run_rf_train_test.py').is_file())
runs = {}
manifests = {}
metrics_compare = ['train_macro_f1', 'test_macro_f1', 'train_balanced_accuracy', 'test_balanced_accuracy']
pair_keys = ['task', 'subject_id', 'stage']
for trees, dirname in [(20, 'exp005_rf_cumulative_train_test_light20'), (100, 'exp005_rf_cumulative_train_test_100')]:
    folder = comparison_repo / 'outputs' / dirname
    manifest = json.loads((folder / 'run_info.json').read_text(encoding='utf-8'))
    verified = json.loads((folder / 'verification.json').read_text(encoding='utf-8'))
    assert verified['status'] == 'passed' and verified['scope'] == 'main'
    assert verified['n_folds'] == 90 and verified['n_test_subjects'] == 15
    assert verified['signature'] == manifest['signature']
    data = pd.read_csv(folder / 'metrics.csv')
    assert len(data) == 90 and not data.duplicated(pair_keys).any()
    assert set(data.task) == {'binary', 'multiclass'} and set(data.stage) == {'A', 'A+B', 'A+B+C'}
    assert data[metrics_compare].notna().all().all()
    assert all(set(g.subject_id) == set(manifest['config']['expected_subjects'])
               for _, g in data.groupby(['task', 'stage']))
    assert all(s['params']['n_estimators'] == trees for s in manifest['config']['stages'])
    data['gap_macro_f1'] = data.train_macro_f1 - data.test_macro_f1
    data['gap_balanced_accuracy'] = data.train_balanced_accuracy - data.test_balanced_accuracy
    runs[trees], manifests[trees] = data, manifest

for field in ['features', 'schema', 'feature_sha256', 'expected_subjects', 'tasks', 'seed']:
    assert manifests[20]['config'][field] == manifests[100]['config'][field]
for field in ['schema_sha256', 'source_sha256', 'environment']:
    assert manifests[20][field] == manifests[100][field]
for old, new in zip(manifests[20]['config']['stages'], manifests[100]['config']['stages']):
    assert old['name'] == new['name']
    old_params = {'max_depth': None, 'max_features': 'sqrt', **old['params']}
    new_params = {'max_depth': None, 'max_features': 'sqrt', **new['params']}
    old_params.pop('n_estimators'); new_params.pop('n_estimators')
    assert old_params == new_params

paired = runs[20].merge(runs[100], on=pair_keys, suffixes=('_20', '_100'), validate='one_to_one')
assert len(paired) == 90
for count in ['n_train', 'n_test']:
    assert paired[f'{count}_20'].eq(paired[f'{count}_100']).all()
comparison = paired.groupby(['task', 'stage'])[[f'{metric}_{trees}' for metric in metrics_compare for trees in [20, 100]]].mean()
for metric in metrics_compare:
    comparison[f'delta_{metric}'] = comparison[f'{metric}_100'] - comparison[f'{metric}_20']
print('Đã kiểm tra hai lượt đủ 90 fold, cùng dữ liệu và cấu hình ngoài số cây.')
display(comparison[[f'{metric}_{suffix}' if suffix in ['20', '100'] else f'delta_{metric}'
                    for metric in ['test_macro_f1', 'test_balanced_accuracy']
                    for suffix in ['20', '100', 'delta']]].round(4))

Đã kiểm tra hai lượt đủ 90 fold, cùng dữ liệu và cấu hình ngoài số cây.


test_macro_f1_20  test_macro_f1_100  delta_test_macro_f1  test_balanced_accuracy_20  test_balanced_accuracy_100  delta_test_balanced_accuracy
task       stage                                                                                                                                               
binary     A                0.8678             0.8735               0.0057                     0.8756                      0.8865                        0.0109
           A+B              0.8624             0.8749               0.0125                     0.8711                      0.8864                        0.0153
           A+B+C            0.8813             0.8922               0.0110                     0.8933                      0.9007                        0.0073
multiclass A                0.6278             0.6480               0.0202                     0.6445                      0.6700                        0.0255
           A+B              0.6414             0.6320              -0.0094                     0.6641                      0.6561                       -0.0079
           A+B+C            0.6147             0.6226               0.0079                     0.6477                      0.6398                       -0.0079

## Nhận xét: tăng lên 100 cây có thay đổi gì?

| Bài toán | Cấu hình | Test F1: 20 → 100 | Δ F1 | Test BA: 20 → 100 | Δ BA |
| --- | --- | ---: | ---: | ---: | ---: |
| Nhị phân | A | 0.8678 → 0.8735 | +0.0057 | 0.8756 → 0.8865 | +0.0109 |
| Nhị phân | A+B | 0.8624 → 0.8749 | +0.0125 | 0.8711 → 0.8864 | +0.0153 |
| Nhị phân | A+B+C | 0.8813 → 0.8922 | +0.0110 | 0.8933 → 0.9007 | +0.0073 |
| Ba lớp | A | 0.6278 → 0.6480 | +0.0202 | 0.6445 → 0.6700 | +0.0255 |
| Ba lớp | A+B | 0.6414 → 0.6320 | -0.0094 | 0.6641 → 0.6561 | -0.0079 |
| Ba lớp | A+B+C | 0.6147 → 0.6226 | +0.0079 | 0.6477 → 0.6398 | -0.0079 |

**Nhị phân:** cả ba cấu hình tăng cả Macro-F1 và BA khi chuyển từ 20 lên 100 cây. Ở 100 cây, A+B+C có điểm trung bình cao nhất trong ba cấu hình: F1 = 0,8922 và BA = 0,9007. Riêng bước thêm B ở 100 cây chỉ tăng F1 rất ít (A: 0,8735 → A+B: 0,8749), còn BA giảm rất nhẹ (0,8865 → 0,8864). Như vậy, thêm B chưa cải thiện đồng thời cả hai metric.

**Ba lớp:** A tăng F1 từ 0,6278 lên 0,6480 và BA từ 0,6445 lên 0,6700. A+B giảm cả hai metric. A+B+C tăng F1 (0,6147 → 0,6226), nhưng BA giảm (0,6477 → 0,6398). Ở 100 cây, A có cả F1 và BA cao nhất trong ba cấu hình. Cấu hình tốt nhất theo F1 của lượt 20 cây là A+B, còn lượt 100 cây là A.

**Vì sao nhiều cây hơn chưa chắc điểm cao hơn?** RF kết hợp dự đoán từ nhiều cây. Tăng số cây thường giúp giảm ảnh hưởng của từng cây riêng lẻ, nhưng không bảo đảm tăng điểm ở mọi người test hoặc mọi cấu hình. Kết quả này dùng một seed cố định nên chưa chứng minh độ ổn định qua nhiều seed hoặc khác biệt có ý nghĩa thống kê.

### Điểm train–test và thời gian

| Bài toán | Cấu hình | Khoảng cách F1 train − test: 20 cây | 100 cây |
| --- | --- | ---: | ---: |
| Nhị phân | A | 0.1322 | 0.1265 |
| Nhị phân | A+B | 0.1375 | 0.1250 |
| Nhị phân | A+B+C | 0.1184 | 0.1076 |
| Ba lớp | A | 0.3722 | 0.3520 |
| Ba lớp | A+B | 0.3563 | 0.3666 |
| Ba lớp | A+B+C | 0.3822 | 0.3748 |

Điểm train vẫn rất cao: khoảng 0,997–1,000 ở cả hai lượt. Tăng số cây không giải quyết hết khoảng cách train–test. Ở bài toán ba lớp, khoảng cách vẫn lớn (khoảng 0,35–0,38); cần xem xét khả năng khái quát sang người mới và khác biệt giữa người, thay vì kết luận chỉ cần tăng cây là hết overfitting.

Tổng cột `seconds` của 90 fold: **20 cây ≈ 63.4 giây**, **100 cây ≈ 280.8 giây**, tức khoảng **4.43 lần**. Cột này đo fit và dự đoán train/test trong từng fold; không phải toàn bộ thời gian chạy notebook, cũng không phải độ trễ dự đoán một cửa sổ. Hai lượt chạy khác thời điểm nên chỉ dùng để tham khảo chi phí trên máy.

**Kết luận:** giữ kết quả 20 cây làm mốc so sánh và dùng 100 cây làm lượt khảo sát bổ sung. Tăng số cây cải thiện rõ hơn ở nhị phân; ở ba lớp hiệu quả phụ thuộc cấu hình. Chưa chọn cấu hình triển khai chỉ từ bảng test này; chọn tham số bằng inner CV thuộc EXP-009.

## Kết luận và hướng xử lý tiếp theo

**Tóm lại:** tăng từ 20 lên 100 cây giúp cả ba cấu hình tốt hơn ở bài toán nhị phân, nhưng chưa giải quyết được bài toán ba lớp và khoảng cách train–test. Vì vậy, hướng tiếp theo là kiểm soát độ phức tạp của cây và đánh giá khả năng tổng quát sang người mới, thay vì chỉ tiếp tục tăng số cây.

### Kết quả rút ra từ EXP-005

- Giữ **RF mặc định 100 cây (A)** làm baseline chính để đối chiếu các thí nghiệm tiếp theo. Giữ kết quả 20 cây để minh họa tác động của số cây.
- Với **nhị phân**, A+B+C ở 100 cây là cấu hình đáng khảo sát tiếp: test Macro-F1 = 0,8922 và BA = 0,9007, cao nhất trong ba cấu hình đã thử. Đây là ứng viên; chưa chốt làm mô hình cuối chỉ từ điểm test LOSO này.
- Với **ba lớp**, A ở 100 cây có cả hai metric cao nhất trong ba cấu hình. Việc giới hạn độ sâu ở 8 và thêm `max_features=0.25` chưa đem lại cải thiện so với A trong lượt 100 cây. Không áp dụng nguyên cấu hình tốt cho nhị phân sang ba lớp.

### Hướng cải thiện

1. **Kiểm soát cây bằng EXP-009:** khảo sát `max_depth`, `min_samples_leaf` và `max_features`, có thể giữ 100 cây trong lượt khảo sát để tập trung vào tác động của các tham số này. Ví dụ, tăng `min_samples_leaf` buộc mỗi lá chứa nhiều mẫu hơn, có thể giảm việc học quá sát dữ liệu train. Đây là giả thuyết cần kiểm tra, không bảo đảm điểm test tăng.
2. **Chọn tham số bằng nested LOSO:** ở mỗi fold ngoài, giữ riêng 1 người test; chỉ dùng 14 người train để chọn tham số bằng các fold trong chia theo người. Sau đó huấn luyện trên 14 người với tham số đã chọn và đánh giá 1 lần trên người test. Không dùng kết quả của người test để chọn tham số cho chính fold đó.
3. **Tìm nguyên nhân ở bài toán ba lớp:** xem confusion matrix và F1 từng lớp, từng người để biết lớp nào bị nhầm và những người nào có điểm thấp. Nếu có dấu hiệu thiên lệch theo lớp, khảo sát cân bằng lớp trong EXP-014; mọi xử lý cân bằng chỉ thực hiện trên tập train của từng fold.
4. **Đánh giá lại trên cùng giao thức:** so với baseline A 100 cây bằng Macro-F1, BA, độ phân tán qua người và khoảng cách train–test. Điểm train giảm chưa đủ để kết luận tốt hơn; cần xem điểm test có cải thiện hay không. Có thể chạy thêm vài seed để xem xu hướng có lặp lại.

**Kết luận chung:** Giữ RF 100 cây làm baseline. Tăng số cây cải thiện kết quả nhị phân nhưng chưa giải quyết được khả năng tổng quát ở bài toán ba lớp. Bước tiếp theo là chọn tham số kiểm soát độ phức tạp bằng nested LOSO, đồng thời phân tích lỗi theo lớp và theo người để xác định hướng cải thiện.
